# Dental Clinic Reception Assistant — Gemini API

This notebook follows the structure of the supplied Gemini end-to-end example, but adapts it for a dental-clinic reception assistant.

**Important:** the original notebook contained a hard-coded API key. This version does not. Set `GEMINI_API_KEY` as an environment variable before running the notebook.


In [2]:
# Install once if needed:
# !pip install -U google-genai panel python-dotenv

import os
from google import genai
from google.genai import types
import panel as pn
import utils

pn.extension()
import os
from google import genai

API_KEY = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash"

print("Gemini client ready.")
print("Model:", MODEL)

Gemini client ready.
Model: gemini-2.5-flash


In [3]:
response = client.models.generate_content(
    model=MODEL,
    contents="Hello, reply with OK"
)

print(response.text)

OK


In [4]:
# Test the Gemini connection
response = client.models.generate_content(
    model=MODEL,
    contents="Reply with exactly: Dental reception assistant is ready.",
)
print(response.text)


Dental reception assistant is ready.


In [5]:
SYSTEM_PROMPT = utils.build_system_prompt()

def get_completion_from_messages(messages, model=MODEL, temperature=0.3, max_tokens=500):
    """Gemini wrapper using the current google-genai SDK."""
    contents = []
    for message in messages:
        role = message.get("role", "user")
        content = message.get("content", "")
        if role == "system":
            # System instructions are handled separately below.
            continue
        gemini_role = "model" if role == "assistant" else "user"
        contents.append(types.Content(
            role=gemini_role,
            parts=[types.Part.from_text(text=str(content))],
        ))

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=config,
    )
    return response.text.strip()


In [23]:
response = handle_appointment_request(
    "I want to book a dental appointment. My name is Arun and my phone is 9876543210. I want August 10 at 10 AM for a dental check-up."
)

print(response)

BOOK_APPOINTMENT

Name: Arun
Phone: 9876543210
Date: August 10
Time: 10 AM
Reason: dental check-up


In [7]:
response = client.models.generate_content(
    model=MODEL,
    contents="""
Extract the following appointment details from this message:

I want to book a dental appointment. My name is Arun and my phone is 9876543210. I want August 10 at 10 AM for a dental check-up.

Return ONLY this format:

Name: ...
Phone: ...
Date: ...
Time: ...
Reason: ...
"""
)

print(response.text)

Name: Arun
Phone: 9876543210
Date: August 10
Time: 10 AM
Reason: Dental check-up


In [8]:
import json
import os
from datetime import datetime

APPOINTMENTS_FILE = "appointments.json"


def load_appointments():
    if not os.path.exists(APPOINTMENTS_FILE):
        with open(APPOINTMENTS_FILE, "w", encoding="utf-8") as file:
            json.dump([], file, indent=4)

    with open(APPOINTMENTS_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def save_appointment(name, phone, date, time, reason):
    appointments = load_appointments()

    appointment = {
        "name": name,
        "phone": phone,
        "date": date,
        "time": time,
        "reason": reason,
        "booked_at": datetime.now().isoformat(timespec="seconds")
    }

    appointments.append(appointment)

    with open(APPOINTMENTS_FILE, "w", encoding="utf-8") as file:
        json.dump(appointments, file, indent=4)

    return appointment


def get_all_appointments():
    return load_appointments()


print("Appointment system ready.")

Appointment system ready.


In [9]:
save_appointment(
    name="Arun",
    phone="9876543210",
    date="August 10",
    time="10 AM",
    reason="Dental check-up"
)

print("Arun appointment saved successfully!")

Arun appointment saved successfully!


In [10]:
# Dental appointment booking through chatbot

def handle_appointment_request(user_message):
    prompt = f"""
You are a dental clinic reception assistant.

The patient said:
"{user_message}"

If the patient wants to book an appointment, extract these details:

- name
- phone
- date
- time
- reason

If any detail is missing, politely ask for the missing information.

When all five details are available, reply with:
BOOK_APPOINTMENT

Then provide the extracted details in this exact format:

Name: ...
Phone: ...
Date: ...
Time: ...
Reason: ...

If the patient is NOT trying to book an appointment, reply:
NOT_APPOINTMENT
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text

In [11]:
def book_appointment_from_message(user_message):
    # Get appointment details from Gemini
    response = client.models.generate_content(
        model=MODEL,
        contents=f"""
You are a dental clinic reception assistant.

Extract appointment details from this patient message:

"{user_message}"

Return ONLY these 5 lines:

Name: ...
Phone: ...
Date: ...
Time: ...
Reason: ...

If a detail is missing, write:
Missing
"""
    )

    details = response.text.strip()
    print(details)

    # Convert Gemini response into values
    data = {}

    for line in details.splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            data[key.strip()] = value.strip()

    # Check all required details
    required = ["Name", "Phone", "Date", "Time", "Reason"]

    if any(data.get(key) in [None, "", "Missing"] for key in required):
        return "Please provide all appointment details."

    # Save appointment
    save_appointment(
        name=data["Name"],
        phone=data["Phone"],
        date=data["Date"],
        time=data["Time"],
        reason=data["Reason"]
    )

    return (
        f"✅ Appointment booked successfully!\n\n"
        f"Name: {data['Name']}\n"
        f"Phone: {data['Phone']}\n"
        f"Date: {data['Date']}\n"
        f"Time: {data['Time']}\n"
        f"Reason: {data['Reason']}"
    )

In [12]:
id="test_booking_01"
result = book_appointment_from_message(
    "I want to book a dental appointment. My name is Kumar. My phone number is 9876543210. I need appointment on August 15 at 11 AM for tooth pain."
)

print(result)

Name: Kumar
Phone: 9876543210
Date: August 15
Time: 11 AM
Reason: tooth pain
✅ Appointment booked successfully!

Name: Kumar
Phone: 9876543210
Date: August 15
Time: 11 AM
Reason: tooth pain


In [13]:
# Test appointment saving

test_appointment = save_appointment(
    name="Test Patient",
    phone="9876543210",
    date="10-08-2026",
    time="10:00 AM",
    reason="Dental check-up"
)

print("Appointment saved successfully!")
print(test_appointment)

Appointment saved successfully!
{'name': 'Test Patient', 'phone': '9876543210', 'date': '10-08-2026', 'time': '10:00 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T14:20:43'}


In [14]:
print("Appointment booking connection step started.")

Appointment booking connection step started.


In [15]:
patient_name = input("Enter patient name: ")

print("Patient name:", patient_name)

Enter patient name:  shayin


Patient name: shayin


In [16]:
def show_appointments():
    appointments = get_all_appointments()

    if not appointments:
        return "No appointments found."

    message = "🦷 Appointment List\n\n"

    for i, appt in enumerate(appointments, start=1):

        message += f"""
{i}. Appointment
{appt}
-------------------------
"""

    return message

In [17]:
def cancel_appointment(phone):
    appointments = get_all_appointments()

    if not appointments:
        return "No appointments found."

    found = False
    remaining = []

    for appt in appointments:
        if str(appt.get("phone", "")) == str(phone):
            found = True
        else:
            remaining.append(appt)

    if not found:
        return f"No appointment found for phone number {phone}."

    import json

    with open("appointments.json", "w") as f:
        json.dump(remaining, f, indent=4)

    return f"Appointment for phone number {phone} has been cancelled successfully."

In [18]:
def find_appointment(phone):
    appointments = get_all_appointments()

    if not appointments:
        return "No appointments found."

    for appt in appointments:
        if str(appt.get("phone", "")) == str(phone):
            return f"""
🦷 Appointment Found

{appt}
"""

    return f"No appointment found for phone number {phone}."

In [19]:
import json

with open("appointments.json", "r") as f:
    data = json.load(f)

print(data)

[{'name': 'Arun', 'phone': '9876543210', 'date': 'August 10', 'time': '10 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T13:04:56'}, {'name': 'Kumar', 'phone': '9876543210', 'date': 'August 15', 'time': '11 AM', 'reason': 'tooth pain', 'booked_at': '2026-08-09T13:05:43'}, {'name': 'Test Patient', 'phone': '9876543210', 'date': '10-08-2026', 'time': '10:00 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T13:06:21'}, {'name': 'Arun', 'phone': '9876543210', 'date': 'August 10', 'time': '10 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T14:20:37'}, {'name': 'Kumar', 'phone': '9876543210', 'date': 'August 15', 'time': '11 AM', 'reason': 'tooth pain', 'booked_at': '2026-08-09T14:20:42'}, {'name': 'Test Patient', 'phone': '9876543210', 'date': '10-08-2026', 'time': '10:00 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T14:20:43'}]


In [20]:
appointments = get_all_appointments()
print(appointments[0])

{'name': 'Arun', 'phone': '9876543210', 'date': 'August 10', 'time': '10 AM', 'reason': 'Dental check-up', 'booked_at': '2026-08-09T13:04:56'}


In [21]:
import panel as pn
import re

pn.extension()

context = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

chat_box = pn.Column(
    height=450,
    scroll=True,
    sizing_mode="stretch_width"
)

inp = pn.widgets.TextInput(
    placeholder="Type your question here...",
    sizing_mode="stretch_width"
)

send_button = pn.widgets.Button(
    name="Send",
    button_type="primary"
)


def send_message(event):
    global context

    user_input = inp.value.strip()

    if not user_input:
        return

    chat_box.append(
        pn.pane.Markdown(
            f"**You:** {user_input}"
        )
    )

    inp.value = ""

    try:

        if "show" in user_input.lower() and "appointment" in user_input.lower():

            response = show_appointments()

        elif (
            "find" in user_input.lower()
            or "search" in user_input.lower()
        ):

            phone_match = re.search(
                r'\b\d{10}\b',
                user_input
            )

            if phone_match:
                phone = phone_match.group()
                response = find_appointment(phone)
            else:
                response = "Please provide your 10-digit phone number."

        elif "cancel" in user_input.lower():

            phone_match = re.search(
                r'\b\d{10}\b',
                user_input
            )

            if phone_match:
                phone = phone_match.group()
                response = cancel_appointment(phone)
            else:
                response = "Please provide your 10-digit phone number."

        elif (
            "appointment" in user_input.lower()
            or "book" in user_input.lower()
        ):

            response = book_appointment_from_message(user_input)

        else:

            response = client.models.generate_content(
                model=MODEL,
                contents=f"""
You are a friendly dental clinic reception assistant
for Bright Smile Dental Clinic.

Answer the patient's question clearly and politely.

Patient question:
{user_input}
"""
            ).text

        chat_box.append(
            pn.pane.Markdown(
                f"**🦷 Dental Reception Assistant:** {response}"
            )
        )

    except Exception as e:

        chat_box.append(
            pn.pane.Markdown(
                f"**Error:** {str(e)}"
            )
        )


send_button.on_click(send_message)


dashboard = pn.Column(
    pn.pane.Markdown(
        "# 🦷 Dental Clinic Reception Assistant"
    ),
    chat_box,
    pn.Row(
        inp,
        send_button
    ),
    sizing_mode="stretch_width"
)

dashboard

Column(sizing_mode='stretch_width')
    [0] Markdown(str)
    [1] Column(height=450, scroll=True, sizing_mode='stretch_width')
    [2] Row
        [0] TextInput(placeholder='Type your question h..., sizing_mode='stretch_width')
        [1] Button(button_type='primary', name='Send')

In [22]:
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello to me in one short sentence."
)

print(response.text)

Hello!


In [ ]:
from utils import get_all_appointments

appointments = get_all_appointments()

print(appointments)

In [ ]:
show_appointments()

## Example questions

- What time are you open on Saturday?
- I want to book a dental check-up.
- Do you provide teeth cleaning?
- Where is the clinic?
- I have severe swelling and bleeding. What should I do?

Update the clinic details in `utils.py` before using this with real patients.
